# 04 回测时间轴与新闻时间对齐

## 本课学习目标

- A. 量化金融主线：回测（Backtesting）和未来数据泄漏（Look-Ahead Bias）
零基础解释：回测不能使用当时还不知道的信息。
- B. 大语言模型主线：上下文窗口（Context Window）和 available_at
零基础解释：新闻只有在可用时间后才能进入模型上下文。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：构造正确和错误时间轴，自动拒绝未来新闻。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
import pandas as pd
from learning.src.market_data import load_price_data
from learning.src.time_alignment import monthly_signal_schedule, validate_event_timeline, filter_available_news
prices = load_price_data(DATA / "sample_prices.csv")
news = pd.read_csv(DATA / "sample_news.csv")
schedule = monthly_signal_schedule(prices)
display(schedule.head())
row = schedule.iloc[0]
usable = filter_available_news(news, row["signal_timestamp"])
print("usable news rows:", len(usable))
try:
    validate_event_timeline(news["available_at"].iloc[-1], row["signal_timestamp"], row["execution_timestamp"])
except ValueError as exc:
    print("错误案例被拒绝:", exc)

## 结尾总结

你现在应该理解：回测必须严格按照时间顺序，未来信息是量化策略最危险的错误之一。

本课核心收获：
- 回测在历史数据上模拟策略，必须防止使用当时未知的信息；
- 未来数据泄漏（Look-Ahead Bias）会让回测结果失真（虚高）；
- 新闻的 available_at 通常晚于 published_at（发布→审核→分发有延迟）；
- 收盘后发布的新闻不能用于同日收盘前的信号；
- 周末新闻只能在下一交易日使用；
- 月末收盘后生成信号 → 下一交易日开盘执行；
- Context Window 的边界必须严格遵守时间对齐规则。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Backtesting（回测）
- Look-Ahead Bias（未来数据泄漏）
- Context Window（上下文窗口）
- Signal Timestamp（信号时间戳）
- Execution Timestamp（执行时间戳）
- available_at（可用时间）

### 常见错误

1. **使用收盘后新闻在同日收盘成交**：如果新闻在 17:00 才可用，16:00 收盘时"不可能"看到它。
2. **将 published_at 误认为 available_at**：新闻发布 ≠ 新闻可用，两者通常有时差。
3. **用未来交易日价格生成今天信号**：使用 t+1 的价格来决定 t 日的交易方向属于未来泄漏。
4. **忽略时区**：如果发布时间是 UTC，而交易时间是 ET，未转换时区可能导致一天的时间差。

### 课后练习

1. **找出时间轴错误**：从 sample_news.csv 中找一条 published_at 在收盘后（16:00 之后）的新闻，解释为什么它不能用于同日信号。
2. **验证过滤逻辑**：调用 `filter_available_news()` 并统计被过滤掉的新闻数量，解释被过滤的原因。
3. **构造泄漏案例**：故意将一条明天才可用的新闻用于今天的信号，调用 `validate_event_timeline()`，确认它被拒绝。

下一课与本课有什么关系：下一课在时间对齐的基础上，引入动量因子和提示词工程，教你如何构建量化因子并设计好的提示词。